In [2]:
import tensorflow as tf
from tensorflow.keras import layers

In [3]:
class PolynomialDense(tf.keras.layers.Layer):
    def __init__(self,units,degree=2,use_bias=True,**kwargs):
        super(PolynomialDense,self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self,input_shape):
        input_dim = input_shape[-1]
        
        self.kernels = []
        for d in range(1,self.degree+1):
            w = self.add_weight(
                shape=(input_dim,self.units),
                initializer='glorot_uniform',
                trainable=True,
                name=f'kernel_degree_{d}'
            )
            self.kernels.append(w)

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer='zeros',
                trainable=True,
                name='bias'
            )
    def call(self, inputs):

        #Asumimos que los inputs ya están normalizados en [-1, 1]
        x = inputs

        # --- 2. Generamos la base de Legendre
        # P0(x)
        P0 = tf.ones_like(x)

        if self.degree == 0:
            features = [P0]
        else:
            # P1(x)
            P1 = x
            features = [P1]

            if self.degree > 1:
                Pnm2 = P0
                Pnm1 = P1

                for n in range(2, self.degree + 1):
                    Pn = ((2.0 * n - 1.0) * x * Pnm1 - (n - 1.0) * Pnm2) / n
                    features.append(Pn)

                    Pnm2 = Pnm1
                    Pnm1 = Pn

        # --- 3. Combinación lineal con los kernels
        output = 0.0

        for phi, W in zip(features, self.kernels):
            output += tf.matmul(phi, W)

        if self.use_bias:
            output += self.bias

        return output


In [4]:
import numpy as np
import keras
from keras import ops
from keras import layers
#Trabajaremos con el dataset de iris para comprobar los diferentes resultados.
mnistDataset = keras.datasets.mnist

In [5]:
(x_train, y_train), (x_test, y_test) = mnistDataset.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [6]:
x_train = x_train / 255.0
x_test = x_test / 255.0

x_train = 2.0 * x_train - 1.0
x_test  = 2.0 * x_test  - 1.0


x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)


In [7]:
#Modelo con activación 1relu y nuuestra capa personalizada

inputs1 = keras.Input(shape=(784,))
x1 = PolynomialDense(100, degree=4)(inputs1)
x1 = layers.Activation('relu')(x1)

#x2 = PolynomialDense(100, degree=5)(x1)
#x2 = layers.Activation('relu')(x2)

outputs1 = layers.Dense(10, activation='softmax')(x1)

model1 = keras.Model(inputs=inputs1, outputs=outputs1)

In [8]:
model1.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

In [9]:
model1.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5


C:\Users\nicol\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8000 - loss: 0.7201 - val_accuracy: 0.9236 - val_loss: 0.2620
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9098 - loss: 0.2992 - val_accuracy: 0.9371 - val_loss: 0.2176
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9310 - loss: 0.2299 - val_accuracy: 0.9264 - val_loss: 0.2373
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9415 - loss: 0.1939 - val_accuracy: 0.9446 - val_loss: 0.1905
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9479 - loss: 0.1734 - val_accuracy: 0.9461 - val_loss: 0.1926


In [10]:
#resultados
model1.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 987us/step - accuracy: 0.9462 - loss: 0.1843


[0.1842522919178009, 0.9462000131607056]

In [11]:
print("Accuracy del modelo con activación ReLU: ", model1.evaluate(x_test, y_test, verbose=0)[1])

Accuracy del modelo con activación ReLU:  0.9462000131607056
